# 09 딥러닝 예측 모델 (Phase 1 — 3/3)

**40조건** × **LSTM, Autoformer, N-HiTS, iTransformer**

07·08 결과와 합쳐 **10모델 중 조건별 Best**(MAPE) 선정

## 구현 (공식 모델 코드)

| 모델 | 출처 |
|------|------|
| Autoformer, iTransformer | [thuml/Time-Series-Library](https://github.com/thuml/Time-Series-Library) submodule → `tslib_adapter.py` |
| N-HiTS, LSTM | [Nixtla/neuralforecast](https://github.com/Nixtla/neuralforecast) → `neuralforecast_adapter.py` |

- 조건(type×cluster) 내 **family 다변량 패널** 1회 학습 (SKU별 반복 학습 아님)
- lookback=**16**, horizon=**3주**, `DL_EPOCHS=50`(GPU)

> **논문 vs 실습:** 학위논문에서 iTransformer가 강세였던 설정(긴 lookback·다수 variate·WMAPE)과 다릅니다. Ecuador 실습에서는 **XGBoost가 10모델 Best에 더 자주** 선택되는 것이 자연스러울 수 있습니다.

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cuda (NVIDIA GeForce RTX 5090, 32GB)
시계열: 165
학습 <= 201730 | 검증: [201731, 201732, 201733]
Best 선정 기준: MAPE


### ① DL 모델 실험

In [2]:
# Phase 1 — DL 4종: SBC 20조건 + ML 20조건
cache = DATA_PROCESSED / 'phase1_dl_results.parquet'
if cache.exists():
    results = pd.read_parquet(cache)
    summary, best = summarize_phase1(results)
    print('캐시 로드 |', len(results), 'rows')
else:
    sbc = run_phase1_all(df, feat_df, 'SBC_CLUSTER', 'SBC', models=DL_MODELS)
    ml = run_phase1_all(df, feat_df, 'ML_CLUSTER', 'ML', models=DL_MODELS)
    results = pd.concat([sbc, ml], ignore_index=True)
    summary, best = summarize_phase1(results)
    results.to_parquet(cache, index=False)
    summary.to_csv(DATA_PROCESSED / 'phase1_dl_results_summary.csv', index=False)
    best.to_csv(DATA_PROCESSED / 'phase1_dl_results_best.csv', index=False)
    print('완료 |', len(results), 'rows')

display(best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== 알고리즘별 평균', RANK_METRIC.upper(), '===')
print(results.groupby(['cluster_scheme', 'model'])[RANK_METRIC].mean().unstack('cluster_scheme').round(2))


Phase1 SBC:   0%|          | 0/20 [00:00<?, ?it/s]

Seed set to 42


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


You are using a CUDA device ('NVIDIA GeForce RTX 5090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


C:\Users\kjh\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


완료 | 1320 rows


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
2,ML,A,1,N-HiTS,5713.839964,8344.752088,113.320514,4.755504
6,ML,A,2,N-HiTS,143262.802083,201367.656569,127.743215,4.606288
8,ML,A,3,Autoformer,69530.979513,103684.600459,114.645062,3.776350
14,ML,A,4,N-HiTS,165306.104167,236847.482792,112.459682,3.511581
19,ML,B,1,iTransformer,3547.953429,4964.998610,105.103742,3.234070
21,ML,B,2,LSTM,91334.433083,123133.788125,91.974432,3.050326
27,ML,B,3,iTransformer,42299.177083,66704.110712,70.899822,3.058723
30,ML,C,1,N-HiTS,3700.489823,5247.474260,113.623432,3.344032
33,ML,C,2,LSTM,88947.812500,130693.191539,85.467947,3.834252
38,ML,C,3,N-HiTS,51654.625000,70824.210602,59.559162,4.365904



=== 알고리즘별 평균 MAPE ===
cluster_scheme      ML     SBC
model                         
Autoformer      453.08  298.69
LSTM            132.94  135.61
N-HiTS          121.14  121.84
iTransformer    136.69  137.17


### ② Phase 1 통합 Best (10모델)

In [3]:
stat = pd.read_parquet(DATA_PROCESSED / 'phase1_stat_results.parquet')
ml = pd.read_parquet(DATA_PROCESSED / 'phase1_ml_results.parquet')
dl = pd.read_parquet(DATA_PROCESSED / 'phase1_dl_results.parquet')

phase1_all = pd.concat([stat, ml, dl], ignore_index=True)
phase1_summary, phase1_best = merge_phase1_best(stat, ml, dl)
phase1_all.to_parquet(DATA_PROCESSED / 'phase1_all_results.parquet', index=False)
phase1_summary.to_csv(DATA_PROCESSED / 'phase1_summary.csv', index=False)
phase1_best.to_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv', index=False)
print('Phase1 통합 Best |', len(phase1_best), '조건')
display(phase1_best.sort_values(['cluster_scheme', 'type', 'cluster']))


Phase1 통합 Best | 34 조건


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
8,ML,A,1,XGBoost,4944.263212,6839.595340,95.492940,4.187199
18,ML,A,2,XGBoost,138248.541667,180834.099539,115.117649,4.445066
28,ML,A,3,XGBoost,51902.310242,77426.125331,89.685380,2.873995
35,ML,A,4,RF,147896.034269,199772.436484,95.039133,3.141741
45,ML,B,1,RF,3181.590236,4267.287831,102.193719,3.124449
58,ML,B,2,XGBoost,56637.979000,87948.126476,64.711808,1.891557
68,ML,B,3,XGBoost,36039.312500,54848.326416,58.540456,2.606062
70,ML,C,1,ARIMA,3696.215631,4923.881399,108.596975,3.366768
85,ML,C,2,RF,75887.663677,113124.954524,74.104331,3.271271
93,ML,C,3,N-HiTS,51654.625000,70824.210602,59.559162,4.365904
